In [1]:
import numpy as np
import pandas as pd

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd

# Descargar datos
sp500 = yf.download("^GSPC", start="1950-01-01", auto_adjust=True)

# Precio ajustado
price = sp500['Close']

# Frecuencia mensual (último dato del mes)
price_m = price.resample('ME').last().dropna()

# Log-returns (recomendado)
returns = np.log(price_m).diff().dropna()

print(returns.head())

In [2]:
from config import RAW_DATA_DIR

df_raw = pd.read_csv(RAW_DATA_DIR / "dataset_paper1_monthly.csv",index_col="Date", parse_dates=True)
ffdata_df = df_raw.loc['1926-07-01':'2001-12-01', ['Mkt-RF', 'SMB', 'HML']]

In [3]:
ffdata_df["Mkt-RF"], ffdata_df["SMB"], ffdata_df["HML"], 

(Date
 1926-07-01    2.89
 1926-08-01    2.64
 1926-09-01    0.38
 1926-10-01   -3.27
 1926-11-01    2.54
               ... 
 2001-08-01   -6.40
 2001-09-01   -9.24
 2001-10-01    2.48
 2001-11-01    7.53
 2001-12-01    1.61
 Name: Mkt-RF, Length: 906, dtype: float64,
 Date
 1926-07-01   -2.55
 1926-08-01   -1.14
 1926-09-01   -1.36
 1926-10-01   -0.14
 1926-11-01   -0.11
               ... 
 2001-08-01    2.19
 2001-09-01   -6.23
 2001-10-01    7.49
 2001-11-01   -0.57
 2001-12-01    4.72
 Name: SMB, Length: 906, dtype: float64,
 Date
 1926-07-01   -2.39
 1926-08-01    3.81
 1926-09-01    0.05
 1926-10-01    0.82
 1926-11-01   -0.61
               ... 
 2001-08-01    2.86
 2001-09-01    1.52
 2001-10-01   -7.31
 2001-11-01    2.17
 2001-12-01    0.83
 Name: HML, Length: 906, dtype: float64)

In [4]:
from core.model_selection import (
    IIDNormal, IIDStudent, IIDSkewStudent, IIDGeneralizedError, AR1Normal,
    AR1Student, AR1SkewStudent, GARCH11Normal, GARCH11Student, GARCH11SkewStudent,
    AR1GARCH11Normal, AR1GARCH11Student, AR1GARCH11SkewStudent,
    evaluate_models, summary_table
)

In [14]:
rng = np.random.default_rng(42)

# Define models
models = [
    IIDNormal(),
    IIDStudent(),
    IIDSkewStudent(),
    IIDGeneralizedError(),
    AR1Normal(),
    AR1Student(),
    AR1SkewStudent(),
    GARCH11Normal(),
    GARCH11Student(),
    GARCH11SkewStudent(),
    AR1GARCH11Normal(),
    AR1GARCH11Student(),
    AR1GARCH11SkewStudent(),
]

reports = evaluate_models([ffdata_df["Mkt-RF"], ffdata_df["SMB"], ffdata_df["HML"], returns['^GSPC']], models)

print("=" * 60)
print("GOODNESS-OF-FIT TABLE (sorted by BIC within each series)")
print("=" * 60)
print(summary_table(reports).to_string(index=False))

print("\n" + "=" * 60)
print("BEST MODELS")
print("=" * 60)
for rep in reports:
    flags = rep.diagnostics.flags()
    active = [k for k, v in flags.items() if v]
    print(
        f"Series {rep.series_index} (n={rep.n_obs}): "
        f"BIC→{rep.best_model_bic}  AIC→{rep.best_model_aic}  "
        f"Flags: {active or 'none'}"
    )

GOODNESS-OF-FIT TABLE (sorted by BIC within each series)
 series_idx              model  n_params  n_obs    log_lik        aic        bic       hqic
          0     garch11_skew_t         6    906 -2674.4486  5360.8973  5389.7515  5371.9163
          0 ar1_garch11_skew_t         7    905 -2672.2272  5358.4544  5392.1099  5371.3076
          0          garch11_t         5    906 -2682.3182  5374.6364  5398.6816  5383.8189
          0      ar1_garch11_t         6    905 -2679.3770  5370.7540  5399.6016  5381.7711
          0 ar1_garch11_normal         5    905 -2695.3113  5400.6226  5424.6623  5409.8035
          0     garch11_normal         4    906 -2699.4239  5406.8478  5426.0839  5414.1938
          0         iid_skew_t         4    906 -2735.7290  5479.4580  5498.6942  5486.8040
          0         ar1_skew_t         5    905 -2733.3477  5476.6954  5500.7351  5485.8763
          0              iid_t         3    906 -2742.3728  5490.7457  5505.1728  5496.2552
          0            

[*********************100%***********************]  1 of 1 completed

Ticker         ^GSPC
Date                
1950-02-28  0.009921
1950-03-31  0.004057
1950-04-30  0.038019
1950-05-31  0.044645
1950-06-30 -0.059793
